# 🧵 SareeDrape Studio — Safe First Run (10k × 5 epochs) on Google Colab

**This is the safe first training run.** Use this before the full 135k notebook.

| Setting | Value |
|---|---|
| Images per category | **10,000** (body / saree / blouse) |
| Epochs | **5** |
| Expected time | **40–75 minutes** on Colab T4 GPU |
| Output | `saree_tryon_model.pth` · `model_config.json` · `training_report.json` · `saree_weights.zip` |

---

## ⚠️ Complete these steps BEFORE running

### 1 — Enable GPU
```
Runtime → Change runtime type → Hardware accelerator → GPU (T4)
```

### 2 — Add Kaggle API credentials as Colab Secrets
```
Left sidebar → 🔑 Secrets → + Add new secret
  KAGGLE_USERNAME = your-username
  KAGGLE_KEY      = your-api-key
```
Get key: https://www.kaggle.com/settings → API → Create New Token

### 3 — Accept ALL dataset licences on Kaggle (one-time)
- https://www.kaggle.com/datasets/chirag2466/saree-tryon-dataset
- https://www.kaggle.com/datasets/marquis03/high-resolution-viton-zalando-dataset
- https://www.kaggle.com/datasets/snmahsa/human-images-dataset-men-and-women
- https://www.kaggle.com/datasets/div456/indian-saree-patterns

### 4 — Run all cells
```
Runtime → Run all   (Ctrl+F9)
```

### 5 — Download weights after Cell 11
```
Left sidebar → 📁 Files → /content/saree_weights.zip → right-click → Download
```
Then copy to `backend/dataset/trained_models/` and restart Flask.

---

## Cell 1 — GPU check & install dependencies

**Will raise an error and stop if no GPU is found.** Fix: Runtime → Change runtime type → GPU.

In [ ]:
import subprocess, sys, platform

pkgs = ['opencv-python-headless', 'mediapipe', 'kaggle', 'tqdm', 'Pillow', 'numpy']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
print(f'Python  : {platform.python_version()}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        '\n\n❌ GPU not found!'
        '\nGo to: Runtime → Change runtime type → Hardware accelerator → GPU (T4)'
        '\nThen: Runtime → Run all'
    )

print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print('\n✅ Environment OK — proceed.')

## Cell 2 — Configuration

**Safe first-run settings: 10,000 images per category, 5 epochs.**  
After confirming this works, use `full_saree_training_150k.ipynb` on Kaggle for the full run.

In [ ]:
from pathlib import Path
import json, os, torch

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/content')          # Colab working directory
RAW_DIR     = BASE_DIR / 'dataset' / 'raw'
CLEANED_DIR = BASE_DIR / 'dataset' / 'cleaned'
ANNOT_DIR   = BASE_DIR / 'dataset' / 'annotations'
MODEL_DIR   = BASE_DIR / 'dataset' / 'trained_models'
CKPT_DIR    = MODEL_DIR / 'checkpoints'

for d in [RAW_DIR, CLEANED_DIR, ANNOT_DIR, MODEL_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

WEIGHTS_PATH = MODEL_DIR / 'saree_tryon_model.pth'
CONFIG_PATH  = MODEL_DIR / 'model_config.json'
REPORT_PATH  = MODEL_DIR / 'training_report.json'
ZIP_PATH     = BASE_DIR  / 'saree_weights.zip'

# ── Training parameters ───────────────────────────────────────────────────────
# ⚠️  SAFE FIRST-RUN SETTINGS — do NOT change these for the first run
MAX_PER_CAT  = 10_000   # 10,000 images per category
TOTAL_EPOCHS = 5        # 5 epochs — fast, safe, enough to verify pipeline
BATCH_SIZE   = 16       # reduce to 8 if OOM error
LR           = 1e-4
IMG_H        = 256
IMG_W        = 192
GRID_SIZE    = 5
BASE_CH      = 64
CKPT_EVERY   = 1        # save checkpoint every epoch (only 5 epochs)

DEVICE = torch.device('cuda')

print('━'*50)
print('TRAINING CONFIGURATION')
print('━'*50)
print(f'  Mode          : SAFE FIRST RUN (10k × 5 epochs)')
print(f'  Max per cat   : {MAX_PER_CAT:,} images')
print(f'  Epochs        : {TOTAL_EPOCHS}')
print(f'  Batch size    : {BATCH_SIZE}')
print(f'  Learning rate : {LR}')
print(f'  Image size    : {IMG_H}×{IMG_W}')
print(f'  Device        : {DEVICE} ({torch.cuda.get_device_name(0)})')
print('━'*50)

## Cell 3 — Download datasets from Kaggle

Downloads 4 datasets and routes images to the correct training categories:

| Dataset | Images | → Category |
|---|---|---|
| `chirag2466/saree-tryon-dataset` | 33,564 | `saree_images` + `body_images` |
| `marquis03/high-resolution-viton-zalando-dataset` | 109,432 | `body_images` + `blouse_materials` |
| `snmahsa/human-images-dataset-men-and-women` | ~7,000 | `body_images` (women only, full-body with face) |
| `div456/indian-saree-patterns` | 1,000+ | `blouse_materials` (Ikat/Banarasi/Pichwai/Bandhani fabric) |

Only the first `MAX_PER_CAT` (10,000) images per category are used in training.

In [ ]:
import os, shutil
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

# ── Kaggle authentication ──────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('✅ Credentials loaded from Colab Secrets.')
except Exception as e:
    print(f'⚠️  Colab Secrets unavailable ({e}).')
    print('   Add KAGGLE_USERNAME and KAGGLE_KEY in left sidebar → 🔑 Secrets')

api = KaggleApi()
api.authenticate()
print('✅ Kaggle API authenticated.\n')

# ── Helpers ────────────────────────────────────────────────────────────────────
def count_imgs(folder: Path) -> int:
    if not folder.exists(): return 0
    return sum(1 for f in folder.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)

def copy_imgs(src: Path, dst: Path, label: str = ''):
    dst.mkdir(parents=True, exist_ok=True)
    files = [f for f in src.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
    copied = 0
    for f in files:
        target = dst / f.name
        if not target.exists():
            shutil.copy2(f, target)
            copied += 1
    tag = f'[{label}]' if label else ''
    print(f'  {tag} {copied:,} copied → {dst.name}')
    return copied

def download_ds(slug: str, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    print(f'\n▶ {slug} ...')
    api.dataset_download_files(slug, path=str(dest), unzip=True, quiet=False)
    print(f'  {count_imgs(dest):,} images extracted')
    return dest

def best_subfolder(base: Path, keywords: list) -> Path:
    for kw in keywords:
        for d in sorted(base.rglob('*')):
            if d.is_dir() and kw.lower() in d.name.lower() and count_imgs(d) > 0:
                return d
    candidates = [d for d in base.rglob('*') if d.is_dir() and count_imgs(d) > 0]
    return max(candidates, key=count_imgs) if candidates else base

# ── DS1: chirag2466/saree-tryon-dataset ───────────────────────────────────────
ds1 = download_ds('chirag2466/saree-tryon-dataset', RAW_DIR / 'ds1')
ds1_saree  = best_subfolder(ds1, ['saree', 'cloth', 'garment'])
ds1_person = best_subfolder(ds1, ['person', 'body', 'model', 'human', 'image', 'woman'])
print(f'  saree src  : {ds1_saree.name}  ({count_imgs(ds1_saree):,})')
print(f'  person src : {ds1_person.name} ({count_imgs(ds1_person):,})')
copy_imgs(ds1_saree,  RAW_DIR / 'saree_images', label='DS1→saree')
copy_imgs(ds1_person, RAW_DIR / 'body_images',  label='DS1→body')

# ── DS2: marquis03/high-resolution-viton-zalando-dataset ──────────────────────
# VITON-HD layout: train/image/ + test/image/ → body, train/cloth/ + test/cloth/ → blouse
ds2 = download_ds('marquis03/high-resolution-viton-zalando-dataset', RAW_DIR / 'ds2')
for split in ['train', 'test']:
    if (ds2 / split / 'image').exists():
        copy_imgs(ds2 / split / 'image', RAW_DIR / 'body_images',      label=f'DS2-{split}→body')
    if (ds2 / split / 'cloth').exists():
        copy_imgs(ds2 / split / 'cloth', RAW_DIR / 'blouse_materials',  label=f'DS2-{split}→blouse')

# ── DS3: snmahsa/human-images-dataset-men-and-women ───────────────────────────
# Women full-body with face — ONLY copy women/ folder, skip men/
ds3 = download_ds('snmahsa/human-images-dataset-men-and-women', RAW_DIR / 'ds3')
women_folder = None
for d in ds3.rglob('*'):
    if d.is_dir() and d.name.lower() in ('women', 'woman', 'female') and count_imgs(d) > 0:
        women_folder = d
        break
if not women_folder:
    for d in ds3.rglob('*'):
        if d.is_dir() and 'women' in d.name.lower() and count_imgs(d) > 0:
            women_folder = d
            break
if women_folder:
    print(f'  women folder: {women_folder.name} ({count_imgs(women_folder):,} imgs)')
    copy_imgs(women_folder, RAW_DIR / 'body_images', label='DS3→body(women+face)')
else:
    print('  ⚠ women folder not found in DS3 — skipping to avoid adding men images')

# ── DS4: div456/indian-saree-patterns ─────────────────────────────────────────
# Indian fabric textures: Ikat, Banarasi, Pichwai, Bandhani → blouse_materials
ds4 = download_ds('div456/indian-saree-patterns', RAW_DIR / 'ds4')
copy_imgs(ds4, RAW_DIR / 'blouse_materials', label='DS4→blouse(fabric)')

# ── Summary ────────────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('RAW TOTALS (before cap)')
print('='*55)
for cat in ['saree_images', 'body_images', 'blouse_materials']:
    n = count_imgs(RAW_DIR / cat)
    will_use = min(n, MAX_PER_CAT)
    print(f'  {cat:<22}: {n:>7,}  →  will use {will_use:,}')
print(f'  (capped at MAX_PER_CAT = {MAX_PER_CAT:,})')
print('='*55)

## Cell 4 — Clean & resize images

- Rejects unreadable or tiny images (< 64×64)
- Resizes to **256×192** (Lanczos)
- CLAHE contrast normalisation
- Capped at `MAX_PER_CAT` (10,000) per category

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
MIN_DIM    = 64
CLAHE      = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clean_report = {}

def clean_category(category):
    src = RAW_DIR     / category
    dst = CLEANED_DIR / category
    dst.mkdir(parents=True, exist_ok=True)

    paths = sorted(f for f in src.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
    paths = paths[:MAX_PER_CAT]   # hard cap at 10,000

    saved = rejected = 0
    for fpath in tqdm(paths, desc=f'  {category}', unit='img', ncols=80):
        img = cv2.imread(str(fpath))
        if img is None:
            rejected += 1; continue
        h, w = img.shape[:2]
        if h < MIN_DIM or w < MIN_DIM:
            rejected += 1; continue
        img = cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_LANCZOS4)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        img = cv2.cvtColor(cv2.merge([CLAHE.apply(l), a, b]), cv2.COLOR_LAB2BGR)
        cv2.imwrite(str(dst / (fpath.stem + '.png')), img, [cv2.IMWRITE_PNG_COMPRESSION, 1])
        saved += 1

    clean_report[category] = {'saved': saved, 'rejected': rejected}
    pct = 100 * saved / max(saved + rejected, 1)
    print(f'  {category}: {saved:,} cleaned  {rejected:,} rejected  ({pct:.1f}% pass)')
    return saved

print('Cleaning images (cap = 10,000 per category)...')
for cat in ['saree_images', 'body_images', 'blouse_materials']:
    clean_category(cat)

print('\n' + '='*55)
print('CLEAN SUMMARY')
print('='*55)
for cat, r in clean_report.items():
    print(f"  {cat:<22}: {r['saved']:>6,} saved")
print('='*55)

## Cell 5 — Pose annotations & segmentation masks

- **Body images** → MediaPipe pose keypoints → `annotations/<stem>_keypoints.json`
- **Saree + blouse** → binary mask → `cleaned/masks/<stem>_mask.png`

In [ ]:
import json, cv2
import mediapipe as mp
from tqdm import tqdm

ANNOT_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR = CLEANED_DIR / 'masks'
MASK_DIR.mkdir(parents=True, exist_ok=True)
annot_report = {'body': {'processed': 0, 'failed': 0, 'total': 0}, 'masks': 0}
mp_pose = mp.solutions.pose

# ── Pose keypoints on body images ─────────────────────────────────────────────
body_files = sorted((CLEANED_DIR / 'body_images').glob('*.png'))
annot_report['body']['total'] = len(body_files)
print(f'Annotating {len(body_files):,} body images...')

with mp_pose.Pose(static_image_mode=True, model_complexity=1,
                  min_detection_confidence=0.3) as pose:
    for fpath in tqdm(body_files, desc='  Pose keypoints', unit='img', ncols=80):
        kp_path = ANNOT_DIR / (fpath.stem + '_keypoints.json')
        img = cv2.imread(str(fpath))
        if img is None:
            kp_path.write_text(json.dumps({'keypoints': [], 'error': 'read_failed'}))
            annot_report['body']['failed'] += 1; continue
        try:
            result = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if result.pose_landmarks:
                kps = [{'x': lm.x, 'y': lm.y, 'z': lm.z, 'visibility': lm.visibility}
                       for lm in result.pose_landmarks.landmark]
                kp_path.write_text(json.dumps({'keypoints': kps, 'source': fpath.name}))
                annot_report['body']['processed'] += 1
            else:
                kp_path.write_text(json.dumps({'keypoints': [], 'note': 'no_pose'}))
                annot_report['body']['failed'] += 1
        except Exception as e:
            kp_path.write_text(json.dumps({'keypoints': [], 'error': str(e)}))
            annot_report['body']['failed'] += 1

rate = 100 * annot_report['body']['processed'] / max(len(body_files), 1)
print(f'  Pose: {annot_report["body"]["processed"]:,} OK  '
      f'{annot_report["body"]["failed"]:,} failed  ({rate:.1f}% detection rate)')

# ── Binary masks for saree + blouse ───────────────────────────────────────────
print('\nGenerating masks...')
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
for cat in ['saree_images', 'blouse_materials']:
    for fpath in tqdm(sorted((CLEANED_DIR / cat).glob('*.png')),
                      desc=f'  {cat}', unit='img', ncols=80):
        img = cv2.imread(str(fpath))
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        cv2.imwrite(str(MASK_DIR / (fpath.stem + '_mask.png')), mask)
        annot_report['masks'] += 1

(ANNOT_DIR / 'annotation_report.json').write_text(json.dumps(annot_report, indent=2))
print(f'  {annot_report["masks"]:,} masks saved')
print('  annotation_report.json saved')

## Cell 6 — Model architecture (SareeTryOnModel)

Exact copy of `backend/ai_engine/ml_models.py`.  
Runs a forward-pass test at the end to confirm GPU is working correctly.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if normalize: layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout: layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
                  nn.InstanceNorm2d(out_ch), nn.ReLU(inplace=True)]
        if dropout: layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip): return self.block(torch.cat([x, skip], dim=1))

class FeatureEncoder(nn.Module):
    def __init__(self, in_ch=3, base_ch=64):
        super().__init__()
        c = base_ch
        self.enc = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch, c,   4,2,1,bias=False), nn.LeakyReLU(0.2,True)),
            nn.Sequential(nn.Conv2d(c,   c*2,   4,2,1,bias=False), nn.InstanceNorm2d(c*2),  nn.LeakyReLU(0.2,True)),
            nn.Sequential(nn.Conv2d(c*2, c*4,   4,2,1,bias=False), nn.InstanceNorm2d(c*4),  nn.LeakyReLU(0.2,True)),
            nn.Sequential(nn.Conv2d(c*4, c*8,   4,2,1,bias=False), nn.InstanceNorm2d(c*8),  nn.LeakyReLU(0.2,True)),
        ])
    def forward(self, x):
        feats = []
        for layer in self.enc: x = layer(x); feats.append(x)
        return feats

class CorrelationLayer(nn.Module):
    def forward(self, fa, fb):
        B, C, H, W = fa.shape
        fa_flat = fa.view(B, C, H*W).permute(0,2,1)
        fb_flat = fb.view(B, C, H*W)
        return torch.bmm(fa_flat, fb_flat).view(B, H*W, H, W) / (C**0.5)

class ThetaRegressor(nn.Module):
    _POOL_SIZE = 6
    def __init__(self, in_ch, grid_size=5):
        super().__init__()
        self.grid_size = grid_size
        self.pool = nn.AdaptiveAvgPool2d(self._POOL_SIZE)
        flat = in_ch * self._POOL_SIZE * self._POOL_SIZE
        self.net = nn.Sequential(
            nn.Flatten(), nn.Linear(flat, 512), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, grid_size*grid_size*2), nn.Tanh(),
        )
    def forward(self, x): return self.net(self.pool(x)).view(-1, self.grid_size**2, 2)

class SareeWarpingNetwork(nn.Module):
    def __init__(self, grid_size=5, img_h=256, img_w=192, base_ch=64):
        super().__init__()
        self.grid_size = grid_size
        self.img_h, self.img_w = img_h, img_w
        self.person_enc  = FeatureEncoder(in_ch=3+18, base_ch=base_ch)
        self.garment_enc = FeatureEncoder(in_ch=3,    base_ch=base_ch)
        self.corr  = CorrelationLayer()
        feat_h = img_h // 16; feat_w = img_w // 16
        self.theta = ThetaRegressor(feat_h * feat_w, grid_size)
    def _build_grid(self, theta, B, device):
        pts  = theta.view(B, self.grid_size, self.grid_size, 2)
        grid = F.interpolate(pts.permute(0,3,1,2).float(),
                             size=(self.img_h, self.img_w), mode='bilinear', align_corners=True)
        return grid.permute(0,2,3,1)
    def forward(self, person_with_pose, garment, seg_mask=None):
        fp = self.person_enc(person_with_pose)[-1]
        fg = self.garment_enc(garment)[-1]
        if seg_mask is not None:
            sd = F.adaptive_avg_pool2d(seg_mask.float(), fp.shape[-2:])
            fp = fp * (0.5 + 0.5 * sd)
        corr   = self.corr(fp, fg)
        theta  = self.theta(corr)
        grid   = self._build_grid(theta, garment.size(0), garment.device)
        warped = F.grid_sample(garment, grid, mode='bilinear',
                               padding_mode='border', align_corners=True)
        return warped, grid

class TryOnGenerator(nn.Module):
    def __init__(self, in_ch=29, base_ch=64):
        super().__init__()
        c = base_ch
        self.d1 = DownBlock(in_ch, c,   normalize=False)
        self.d2 = DownBlock(c,     c*2)
        self.d3 = DownBlock(c*2,   c*4)
        self.d4 = DownBlock(c*4,   c*8, dropout=0.4)
        self.d5 = DownBlock(c*8,   c*8, dropout=0.4)
        self.d6 = DownBlock(c*8,   c*8, dropout=0.4)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(c*8, c*8, 3,1,1,bias=False), nn.InstanceNorm2d(c*8), nn.ReLU(True),
            nn.Conv2d(c*8, c*8, 3,1,1,bias=False), nn.InstanceNorm2d(c*8), nn.ReLU(True),
        )
        self.u1 = UpBlock(c*16, c*8, dropout=0.4)
        self.u2 = UpBlock(c*16, c*8, dropout=0.4)
        self.u3 = UpBlock(c*16, c*4)
        self.u4 = UpBlock(c*8,  c*2)
        self.u5 = UpBlock(c*4,  c)
        self.u6 = UpBlock(c*2,  c)
        self.out_head = nn.Sequential(nn.Conv2d(c, 4, 1), nn.Tanh())
    def forward(self, x):
        d1=self.d1(x); d2=self.d2(d1); d3=self.d3(d2)
        d4=self.d4(d3); d5=self.d5(d4); d6=self.d6(d5)
        bot=self.bottleneck(d6)
        u1=self.u1(bot,d6); u2=self.u2(u1,d5); u3=self.u3(u2,d4)
        u4=self.u4(u3,d3);  u5=self.u5(u4,d2); u6=self.u6(u5,d1)
        out=self.out_head(u6)
        rendered=out[:,:3]; alpha=(out[:,3:4]+1)/2
        return rendered, alpha * rendered + (1-alpha) * x[:,:3]

class SareeTryOnModel(nn.Module):
    N_STYLES = 16
    def __init__(self, img_h=256, img_w=192, grid_size=5, base_ch=64):
        super().__init__()
        self.img_h, self.img_w = img_h, img_w
        self.style_emb   = nn.Embedding(self.N_STYLES, 64)
        self.style_proj  = nn.Linear(64, img_h * img_w)
        self.saree_warp  = SareeWarpingNetwork(grid_size, img_h, img_w, base_ch)
        self.blouse_warp = SareeWarpingNetwork(grid_size, img_h, img_w, base_ch)
        self.generator   = TryOnGenerator(in_ch=29, base_ch=base_ch)
    def forward(self, person, saree, blouse, pose_heatmap, seg_mask, style_idx):
        style_e   = self.style_emb(style_idx.clamp(0, self.N_STYLES-1))
        style_map = self.style_proj(style_e).view(-1,1,self.img_h,self.img_w)
        pp = torch.cat([person, pose_heatmap], dim=1)
        ws, _ = self.saree_warp( pp, saree,  seg_mask)
        wb, _ = self.blouse_warp(pp, blouse, seg_mask)
        gen_in = torch.cat([person, ws, wb, pose_heatmap, seg_mask, style_map], dim=1)
        rendered, composed = self.generator(gen_in)
        return {'rendered': rendered, 'composed': composed,
                'warped_saree': ws, 'warped_blouse': wb}

# ── Forward-pass test ─────────────────────────────────────────────────────────
model  = SareeTryOnModel(img_h=IMG_H, img_w=IMG_W).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'Model params : {params/1e6:.2f}M')

with torch.no_grad():
    _t   = lambda c: torch.randn(2, c, IMG_H, IMG_W, device=DEVICE)
    _out = model(person=_t(3), saree=_t(3), blouse=_t(3),
                 pose_heatmap=_t(18), seg_mask=_t(1),
                 style_idx=torch.zeros(2, dtype=torch.long, device=DEVICE))
print(f'Forward pass : rendered={_out["rendered"].shape}  ✅')

## Cell 7 — Dataset & DataLoader

Triplet dataset: `(body_image, saree_image, blouse_image)`.  
Training length = `min(body, saree, blouse)` — balanced by category size.

In [ ]:
import cv2, numpy as np
from torch.utils.data import Dataset, DataLoader

class SareeDataset(Dataset):
    def __init__(self, cleaned_dir, img_h, img_w):
        def _list(cat):
            return sorted((cleaned_dir / cat).glob('*.png'))
        self.body   = _list('body_images')
        self.saree  = _list('saree_images')
        self.blouse = _list('blouse_materials')
        self.img_h  = img_h
        self.img_w  = img_w
        self.n      = min(len(self.body), len(self.saree), len(self.blouse))
        if self.n == 0:
            raise ValueError(
                'One or more categories have 0 cleaned images!\n'
                'Check Cell 3 output — at least one dataset download may have failed.'
            )
        print('Dataset triplets:')
        print(f'  body_images      : {len(self.body):,}')
        print(f'  saree_images     : {len(self.saree):,}')
        print(f'  blouse_materials : {len(self.blouse):,}')
        print(f'  → training on    : {self.n:,} triplets')

    def _load(self, fpath):
        img = cv2.imread(str(fpath))
        if img is None:
            img = np.zeros((self.img_h, self.img_w, 3), dtype=np.uint8)
        img = cv2.resize(img, (self.img_w, self.img_h))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return torch.from_numpy(img.astype(np.float32).transpose(2,0,1) / 127.5 - 1.0)

    def __len__(self): return self.n
    def __getitem__(self, idx):
        return (
            self._load(self.body  [idx % len(self.body)]),
            self._load(self.saree [idx % len(self.saree)]),
            self._load(self.blouse[idx % len(self.blouse)]),
        )

dataset    = SareeDataset(CLEANED_DIR, IMG_H, IMG_W)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)
print(f'\nDataLoader : {len(dataloader):,} batches/epoch')
print(f'Total steps: {len(dataloader) * TOTAL_EPOCHS:,}  ({TOTAL_EPOCHS} epochs)')

## Cell 8 — Training loop (5 epochs)

- Optimiser: Adam (β₁=0.5, β₂=0.999)
- Scheduler: CosineAnnealingLR
- Loss: MSE(rendered, body)
- Checkpoint saved every epoch
- Best model continuously saved to `saree_tryon_model.pth`

**Expected: loss ~1.0 → ~0.3–0.6 over 5 epochs. Decreasing trend = pipeline working.**

In [ ]:
import time, datetime

optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.5, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)
criterion = nn.MSELoss()

best_loss  = float('inf')
train_log  = []
started_at = datetime.datetime.utcnow().isoformat()

print(f'Training started : {started_at}')
print(f'Epochs           : {TOTAL_EPOCHS}  (safe first run)')
print(f'Batches/epoch    : {len(dataloader):,}')
print(f'Device           : {torch.cuda.get_device_name(0)}')
print('='*65)

for epoch in range(1, TOTAL_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for step, (body, saree, blouse) in enumerate(dataloader):
        body   = body.to(DEVICE, non_blocking=True)
        saree  = saree.to(DEVICE, non_blocking=True)
        blouse = blouse.to(DEVICE, non_blocking=True)
        B      = body.size(0)

        pose_hm   = torch.zeros(B, 18, IMG_H, IMG_W, device=DEVICE)
        seg_mask  = torch.ones( B,  1, IMG_H, IMG_W, device=DEVICE)
        style_idx = torch.randint(0, SareeTryOnModel.N_STYLES, (B,), device=DEVICE)

        out  = model(person=body, saree=saree, blouse=blouse,
                     pose_heatmap=pose_hm, seg_mask=seg_mask,
                     style_idx=style_idx)
        loss = criterion(out['rendered'], body)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

        if step % 100 == 0:
            pct = 100 * step / len(dataloader)
            mem = torch.cuda.memory_allocated() / 1e9
            print(f'  e{epoch}/{TOTAL_EPOCHS} [{pct:5.1f}%] '
                  f'step {step:4d}/{len(dataloader)}  '
                  f'loss={loss.item():.6f}  '
                  f'gpu={mem:.1f}GB')

    scheduler.step()
    avg_loss = epoch_loss / len(dataloader)
    elapsed  = time.time() - t0
    lr_now   = scheduler.get_last_lr()[0]
    remaining = elapsed * (TOTAL_EPOCHS - epoch)

    print(f'\n✔ Epoch {epoch}/{TOTAL_EPOCHS}  '
          f'avg_loss={avg_loss:.6f}  '
          f'lr={lr_now:.2e}  '
          f'time={elapsed/60:.1f}min  '
          f'remaining≈{remaining/60:.0f}min\n')

    train_log.append({'epoch': epoch, 'loss': round(avg_loss, 6),
                      'lr': lr_now, 'elapsed_s': round(elapsed, 1)})

    # ── Save checkpoint every epoch ───────────────────────────────────────────
    ckpt_path = CKPT_DIR / f'checkpoint_epoch{epoch:02d}.pth'
    torch.save({
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': round(avg_loss, 6), 'stub': False,
    }, str(ckpt_path))
    print(f'  ✔ Checkpoint saved → {ckpt_path.name}')

    # ── Save best model ───────────────────────────────────────────────────────
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch':         epoch,
            'model_state':   model.state_dict(),
            'loss':          round(best_loss, 6),
            'stub':          False,
            'is_prototype':  False,
            'training_mode': 'colab_10k',
            'trained_at':    started_at,
            'total_imgs':    len(dataset),
            'device':        str(DEVICE),
            'img_h':         IMG_H,
            'img_w':         IMG_W,
            'max_per_cat':   MAX_PER_CAT,
            'datasets': [
                'chirag2466/saree-tryon-dataset',
                'marquis03/high-resolution-viton-zalando-dataset',
                'snmahsa/human-images-dataset-men-and-women',
                'div456/indian-saree-patterns',
            ],
        }, str(WEIGHTS_PATH))
        print(f'  ✔ Best model updated (loss={best_loss:.6f})')

print('='*65)
print(f'Training complete. Best loss: {best_loss:.6f}')
print('Proceed to Cell 9 to save config and report.')

## Cell 9 — Save model_config.json & training_report.json

In [ ]:
import datetime, json

finished_at = datetime.datetime.utcnow().isoformat()
size_mb     = WEIGHTS_PATH.stat().st_size / 1024**2 if WEIGHTS_PATH.exists() else 0

# ── model_config.json ─────────────────────────────────────────────────────────
config = {
    'img_h':         IMG_H,
    'img_w':         IMG_W,
    'grid_size':     GRID_SIZE,
    'base_ch':       BASE_CH,
    'epochs':        TOTAL_EPOCHS,
    'device':        str(DEVICE),
    'stub':          False,
    'is_prototype':  False,
    'training_mode': 'colab_10k',
    'trained_at':    started_at,
    'finished_at':   finished_at,
    'total_imgs':    len(dataset),
    'best_loss':     round(best_loss, 6),
    'max_per_cat':   MAX_PER_CAT,
    'note':          'Safe first run — 10k images × 5 epochs on Colab T4 GPU',
    'datasets': [
        {'slug': 'chirag2466/saree-tryon-dataset',                  'images': 33564},
        {'slug': 'marquis03/high-resolution-viton-zalando-dataset', 'images': 109432},
        {'slug': 'snmahsa/human-images-dataset-men-and-women',      'images': 7000},
        {'slug': 'div456/indian-saree-patterns',                    'images': 1000},
    ],
    'upgrade_path': 'Use notebooks/full_saree_training_150k.ipynb on Kaggle for full 135k run',
}
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print('✔ model_config.json saved')

# ── training_report.json ──────────────────────────────────────────────────────
report = {
    'training_mode':   'colab_10k',
    'device':          str(DEVICE),
    'gpu':             torch.cuda.get_device_name(0),
    'total_epochs':    TOTAL_EPOCHS,
    'best_loss':       round(best_loss, 6),
    'started_at':      started_at,
    'finished_at':     finished_at,
    'max_per_cat':     MAX_PER_CAT,
    'dataset_sizes': {
        'body_images':      len(dataset.body),
        'saree_images':     len(dataset.saree),
        'blouse_materials': len(dataset.blouse),
    },
    'clean_report':    clean_report,
    'annot_report':    annot_report,
    'epoch_log':       train_log,
    'weights_size_mb': round(size_mb, 2),
    'model_params_m':  round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    'hyperparams': {
        'batch_size': BATCH_SIZE, 'lr': LR,
        'img_h': IMG_H, 'img_w': IMG_W,
    },
    'next_step': 'If loss decreased: deploy weights. Then run full_saree_training_150k.ipynb on Kaggle for production model.',
}
REPORT_PATH.write_text(json.dumps(report, indent=2))
print('✔ training_report.json saved')

print(f'\nOutput files:')
print(f'  saree_tryon_model.pth   {size_mb:.1f} MB')
print(f'  model_config.json')
print(f'  training_report.json')

## Cell 10 — Verify weights

Reloads the saved checkpoint and runs a forward pass to confirm weights are valid and deployable.

In [ ]:
print('Verifying saved weights...')
ckpt = torch.load(str(WEIGHTS_PATH), map_location='cpu')

assert not ckpt.get('stub', True),         '✗ stub flag must be False'
assert not ckpt.get('is_prototype', True), '✗ is_prototype must be False'
assert 'model_state' in ckpt,             '✗ model_state key missing'

verify_model = SareeTryOnModel(img_h=IMG_H, img_w=IMG_W)
verify_model.load_state_dict(ckpt['model_state'])
verify_model.eval()

with torch.no_grad():
    _t  = lambda c: torch.randn(1, c, IMG_H, IMG_W)
    out = verify_model(
        person       = _t(3),
        saree        = _t(3),
        blouse       = _t(3),
        pose_heatmap = _t(18),
        seg_mask     = _t(1),
        style_idx    = torch.tensor([0]),
    )

print(f'✔ rendered shape  : {out["rendered"].shape}')
print(f'✔ epoch trained   : {ckpt["epoch"]}')
print(f'✔ best loss       : {ckpt["loss"]}')
print(f'✔ stub            : {ckpt.get("stub")}')
print(f'✔ is_prototype    : {ckpt.get("is_prototype")}')
print(f'✔ training_mode   : {ckpt.get("training_mode")}')
print(f'✔ max_per_cat     : {ckpt.get("max_per_cat")}')
print('\n✅ Weights valid — proceed to Cell 11 to zip and download.')

## Cell 11 — Zip outputs & download

Creates `saree_weights.zip` containing the 3 output files + all epoch checkpoints.

**After running:** left sidebar → 📁 Files → `/content/saree_weights.zip` → right-click → **Download**

In [ ]:
import zipfile

files_to_zip = [
    (WEIGHTS_PATH, 'saree_tryon_model.pth'),
    (CONFIG_PATH,  'model_config.json'),
    (REPORT_PATH,  'training_report.json'),
]
for ckpt_file in sorted(CKPT_DIR.glob('checkpoint_epoch*.pth')):
    files_to_zip.append((ckpt_file, f'checkpoints/{ckpt_file.name}'))

print(f'Creating {ZIP_PATH.name} ...')
with zipfile.ZipFile(str(ZIP_PATH), 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for src, arcname in files_to_zip:
        if src.exists():
            zf.write(str(src), arcname)
            print(f'  + {arcname:<48} {src.stat().st_size/1024**2:.1f} MB')
        else:
            print(f'  ⚠ MISSING: {arcname}')

zip_mb = ZIP_PATH.stat().st_size / 1024**2
print(f'\n✅ {ZIP_PATH.name}  →  {zip_mb:.1f} MB')
print()
print('━'*60)
print('DOWNLOAD')
print('━'*60)
print('Option A — Files panel (easiest):')
print('  Left sidebar → 📁 Files → /content/saree_weights.zip')
print('  Right-click → Download')
print()
print('Option B — Code download (run this in a new cell):')
print('  from google.colab import files')
print('  files.download("/content/saree_weights.zip")')
print()
print('━'*60)
print('DEPLOY LOCALLY')
print('━'*60)
print('1. Extract saree_weights.zip')
print('2. Copy to project:')
print('     saree_tryon_model.pth  →  backend/dataset/trained_models/')
print('     model_config.json      →  backend/dataset/trained_models/')
print('3. Restart Flask:  cd backend && python app.py')
print('4. Open http://localhost:3000 → Customize → Step 4 → Test Inference')
print('5. Expected: engine = ml_model  (not geometric fallback)')
print('━'*60)